# WorkspaceAI — End-to-End Test Data Seeder

Pushes **realistic Outlook + Teams test data** directly into SeaweedFS and LanceDB
using the existing `ChatDocumentIndexer` pipeline — **zero changes to main app code**.

### Coverage
| Feature | Outlook scenarios | Teams scenarios |
|---|---|---|
| Priority ranking (critical/high/medium/low) | ✅ 12 threads | — |
| Deadline + task extraction | ✅ 8 threads | — |
| Digest summaries | ✅ All threads | ✅ 7 channels |
| Unanswered question tracker | — | ✅ 24 genuine questions |
| Rhetorical question filtering | — | ✅ 18 fake questions |
| Channel filter | — | ✅ Across named channels |
| Multi-turn conversation memory | ✅ Thread chains | ✅ Channel history |
| Urgency signal extraction | ✅ 6 urgency scenarios | ✅ 5 urgency levels |

**Run cells top to bottom.** Cell 7 prints a verification report at the end.

## Cell 1 — Setup: paths, imports, connections

In [ ]:
import sys, os, asyncio, json, hashlib, logging
from datetime import datetime, timedelta, timezone
from pathlib import Path

# ── Point to your app root ────────────────────────────────────────────────────
APP_ROOT = Path("app")          # adjust if notebook is elsewhere
sys.path.insert(0, str(APP_ROOT.resolve()))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(name)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("test_seeder")

# ── Imports from your app ─────────────────────────────────────────────────────
import lancedb
from config                   import LANCEDB_PATH, SEAWEED_FILER, OPENROUTER_KEY
from storage.lancedb_store    import LanceDBStore
from storage.seaweed          import SeaweedStore
from ingestion.indexer        import OpenRouterEmbedder
from connectors.chat_indexer  import ChatDocumentIndexer
from connectors.models        import ConversationDoc, MessageTurn

# ── Connections ───────────────────────────────────────────────────────────────
db_conn  = lancedb.connect(LANCEDB_PATH)
store    = LanceDBStore(db_conn)
seaweed  = SeaweedStore("http://localhost:8888")
embedder = OpenRouterEmbedder(api_key="")
indexer  = ChatDocumentIndexer(store=store, embedder=embedder, seaweed=seaweed)

log.info("✓ Connections ready — LanceDB: %s  SeaweedFS: %s", LANCEDB_PATH, SEAWEED_FILER)
print("✓ All connections ready")

## Cell 2 — Timestamp helpers (used throughout the data)

In [ ]:
def ts(hours_ago: float, mins_offset: int = 0) -> str:
    """Return ISO-8601 UTC timestamp N hours ago (+ optional minute offset)."""
    t = datetime.now(timezone.utc) - timedelta(hours=hours_ago, minutes=mins_offset)
    return t.strftime("%Y-%m-%dT%H:%M:%S")

def msg(mid, sender_id, sender_name, content, hours_ago, mins=0, thread_id=None):
    """Shorthand MessageTurn factory."""
    return MessageTurn(
        message_id  = mid,
        sender_id   = sender_id,
        sender_name = sender_name,
        content     = content,
        sent_at     = ts(hours_ago, mins),
        thread_id   = thread_id,
    )

print("✓ Helpers ready  |  Now: ", ts(0))

## Cell 3 — Outlook Test Threads (12 threads, full priority spectrum)

Each thread maps to one `ConversationDoc` with `platform="outlook"`.
`channel_name` = email subject.  `participants` = sender/recipient IDs.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  OUTLOOK THREADS
#  Priority band targets:
#    CRITICAL (9-10) : board-deck, production-down, investor-call
#    HIGH     (7-8)  : client-escalation, budget-approval, security-breach
#    MEDIUM   (5-6)  : hiring-decision, quarterly-review, cross-team-dep
#    LOW      (1-4)  : newsletter, it-maintenance, party-invitation
# ════════════════════════════════════════════════════════════════════════════

OUTLOOK_THREADS = []

# ── 1. CRITICAL — Board deck deadline ────────────────────────────────────────
OUTLOOK_THREADS.append(ConversationDoc(
    platform      = "outlook",
    chat_id       = "thread_board_deck_001",
    participants  = ["ceo@acme.com", "cfo@acme.com", "you@acme.com"],
    display_names = {"ceo@acme.com": "Sarah Chen (CEO)", "cfo@acme.com": "Michael Torres (CFO)", "you@acme.com": "You"},
    channel_name  = "URGENT: Board Presentation — Final Deck Needed by EOD Today",
    fetched_at    = ts(0),
    messages      = [
        msg("bd001", "ceo@acme.com", "Sarah Chen", "The board meeting is tomorrow at 9 AM sharp. We still don't have the final revenue slide from Q3 actuals. Michael — can you confirm the numbers and send the updated slide to me by 5 PM TODAY? This is absolutely critical. The investors will be looking at this slide first.", 6),
        msg("bd002", "cfo@acme.com", "Michael Torres", "Sarah — I'm pulling the Q3 numbers now. There is a discrepancy between the regional breakdowns I need to reconcile first. I'll have a draft to you by 3 PM but I need confirmation from the regional leads. Tagging you here.", 5, 30),
        msg("bd003", "ceo@acme.com", "Sarah Chen", "Michael — we cannot miss this deadline. This goes to print at 6 PM. Please confirm by 2:30 PM that the numbers are locked. Also we need the EBITDA comparison chart — please include that. ASAP.", 4, 15),
    ]
))

# ── 2. CRITICAL — Production outage ──────────────────────────────────────────
OUTLOOK_THREADS.append(ConversationDoc(
    platform      = "outlook",
    chat_id       = "thread_prod_outage_002",
    participants  = ["oncall@acme.com", "vp-eng@acme.com", "you@acme.com", "sre@acme.com"],
    display_names = {"oncall@acme.com": "On-Call Engineer", "vp-eng@acme.com": "Priya Sharma (VP Eng)", "you@acme.com": "You", "sre@acme.com": "SRE Team"},
    channel_name  = "P0 INCIDENT: Payment Service Down — 3,000 Customers Affected",
    fetched_at    = ts(0),
    messages      = [
        msg("po001", "oncall@acme.com", "On-Call Engineer", "CRITICAL: Payment service has been returning 503 errors since 02:14 UTC. Approximately 3,000 customers unable to complete checkout. Revenue impact: ~$180K/hour. Root cause: database connection pool exhausted. I need immediate approval to scale the RDS instance — this requires sign-off from VP level.", 3),
        msg("po002", "sre@acme.com", "SRE Team", "Confirmed. We have paged the DB team. ETA on fix is 45 minutes if we can scale. We've already implemented a partial workaround routing 30% of traffic to the backup cluster. Still need VP approval to scale RDS — this will cost ~$800/month extra.", 2, 40),
        msg("po003", "vp-eng@acme.com", "Priya Sharma", "I'm approving the RDS scale immediately. Do it now. Please send me a status update every 15 minutes until resolved. Also — who is doing the customer communication? We need an email out to affected customers in the next 30 minutes. This is P0. All hands.", 2, 20),
    ]
))

# ── 3. CRITICAL — Investor call prep ─────────────────────────────────────────
OUTLOOK_THREADS.append(ConversationDoc(
    platform      = "outlook",
    chat_id       = "thread_investor_003",
    participants  = ["bd@acme.com", "ceo@acme.com", "you@acme.com"],
    display_names = {"bd@acme.com": "James Liu (BD)", "ceo@acme.com": "Sarah Chen (CEO)", "you@acme.com": "You"},
    channel_name  = "Series B Call — Sequoia Partner Call Thursday 10 AM — Need Updated Deck",
    fetched_at    = ts(0),
    messages      = [
        msg("inv001", "bd@acme.com", "James Liu", "The Sequoia call is confirmed for Thursday 10 AM PST. I've shared the previous deck but they specifically asked for updated ARR figures, churn rate for last quarter, and the new GTM roadmap slide. Sarah — can you review and approve by Wednesday EOD? They'll have done their homework before the call.", 20),
        msg("inv002", "ceo@acme.com", "Sarah Chen", "James — I've reviewed. I need the updated ARR from Finance before I can approve. Can you chase that? Also the GTM roadmap is not ready — Marketing needs another day. We may need to delay the call. What's the latest we can reschedule to?", 18),
        msg("inv003", "bd@acme.com", "James Liu", "Rescheduling would be very risky — this is a rare opening in their calendar. I strongly recommend we proceed. I can draft placeholder language for the GTM slide. But we MUST have the ARR numbers confirmed before Wednesday noon. Please escalate to Finance now.", 16),
    ]
))

# ── 4. HIGH — Client escalation ───────────────────────────────────────────────
OUTLOOK_THREADS.append(ConversationDoc(
    platform      = "outlook",
    chat_id       = "thread_client_esc_004",
    participants  = ["client.cto@bigcorp.com", "cs@acme.com", "you@acme.com"],
    display_names = {"client.cto@bigcorp.com": "David Park (BigCorp CTO)", "cs@acme.com": "Customer Success", "you@acme.com": "You"},
    channel_name  = "RE: Unacceptable API Downtime — Escalating to Executive Team",
    fetched_at    = ts(0),
    messages      = [
        msg("ce001", "client.cto@bigcorp.com", "David Park", "We've experienced three separate API outages in the past two weeks, each lasting 20-45 minutes. Our team has escalated this twice already with no resolution. This is completely unacceptable for a platform we pay $180K/year for. I'm escalating directly to your executive team. If we don't see a credible resolution plan by Friday, we will begin evaluating alternatives.", 28),
        msg("ce002", "cs@acme.com", "Customer Success", "Hi David — I sincerely apologise. I've escalated this to our VP of Engineering and our SRE team lead. We're preparing an RCA document and a remediation plan. Can we schedule a call Thursday at 2 PM to walk you through this personally?", 26),
        msg("ce003", "client.cto@bigcorp.com", "David Park", "Thursday works only if you can provide the RCA in writing BEFORE the call. I need something to show my leadership team. Our patience is exhausted. Please confirm.", 24),
    ]
))

# ── 5. HIGH — Budget approval ─────────────────────────────────────────────────
OUTLOOK_THREADS.append(ConversationDoc(
    platform      = "outlook",
    chat_id       = "thread_budget_005",
    participants  = ["finance@acme.com", "dept-head@acme.com", "you@acme.com"],
    display_names = {"finance@acme.com": "Finance Team", "dept-head@acme.com": "Ana Kovac (Dept Head)", "you@acme.com": "You"},
    channel_name  = "Q1 Budget Approval — Deadline Friday 5 PM — $2.4M Engineering Headcount",
    fetched_at    = ts(0),
    messages      = [
        msg("bu001", "finance@acme.com", "Finance Team", "This is a reminder that all Q1 budget submissions must be finalised and approved by Friday 5 PM. Any submission after this deadline will roll into Q2. The Engineering headcount request for $2.4M is currently sitting with department head approval pending. Please action this urgently.", 36),
        msg("bu002", "dept-head@acme.com", "Ana Kovac", "I have reviewed the headcount plan. I'm approving 8 of the 10 positions. The two Senior ML Engineer roles need further justification before I can approve the additional $480K. Can you send me the revised JD and the business case by Thursday noon?", 30),
        msg("bu003", "finance@acme.com", "Finance Team", "Ana — noted. To be clear: Thursday noon is the absolute latest. We cannot process partial approvals — the full submission must be complete by Friday 5 PM or the entire request moves to Q2. Please prioritise.", 28),
    ]
))

# ── 6. HIGH — Security breach ─────────────────────────────────────────────────
OUTLOOK_THREADS.append(ConversationDoc(
    platform      = "outlook",
    chat_id       = "thread_security_006",
    participants  = ["security@acme.com", "cto@acme.com", "you@acme.com", "legal@acme.com"],
    display_names = {"security@acme.com": "Security Team", "cto@acme.com": "Alex Ramos (CTO)", "you@acme.com": "You", "legal@acme.com": "Legal"},
    channel_name  = "CONFIDENTIAL: Potential Data Breach — Immediate Action Required",
    fetched_at    = ts(0),
    messages      = [
        msg("sb001", "security@acme.com", "Security Team", "CONFIDENTIAL — RESTRICTED DISTRIBUTION. Our SIEM flagged anomalous data exfiltration patterns from a contractor account at 11:43 PM last night. Approximately 2,400 customer records may have been accessed without authorisation. We have suspended the account and are conducting forensic analysis. Legal and CTO must be looped in immediately. Do not discuss via any external channel.", 10),
        msg("sb002", "cto@acme.com", "Alex Ramos", "I'm on it. Have you contacted the incident response firm? We need external forensics given the potential scope. Also — what is our GDPR notification obligation timeline? Legal — please advise on the 72-hour clock.", 9, 30),
        msg("sb003", "legal@acme.com", "Legal", "The GDPR 72-hour notification window starts from the moment we have reasonable certainty of a breach. Given what Security has described, I'd treat the clock as running now. We need a legal hold immediately and I need a full briefing before noon today. This cannot wait.", 9),
    ]
))

# ── 7. MEDIUM — Hiring decision ───────────────────────────────────────────────
OUTLOOK_THREADS.append(ConversationDoc(
    platform      = "outlook",
    chat_id       = "thread_hiring_007",
    participants  = ["recruiter@acme.com", "hiring-mgr@acme.com", "you@acme.com"],
    display_names = {"recruiter@acme.com": "Talent Team", "hiring-mgr@acme.com": "Wei Zhang (Hiring Manager)", "you@acme.com": "You"},
    channel_name  = "RE: Staff Engineer Offer — Candidate Deciding by Friday",
    fetched_at    = ts(0),
    messages      = [
        msg("hi001", "recruiter@acme.com", "Talent Team", "Quick update — the candidate for the Staff Engineer role has received a competing offer from Stripe. They've given us until Friday COB to counter. Our current offer is $185K base. Stripe is at $210K + more equity. The hiring manager needs to decide whether we want to go to $200K. The role has been open 4 months.", 48),
        msg("hi002", "hiring-mgr@acme.com", "Wei Zhang", "I want this candidate. Can we go to $195K and sweeten the equity package instead? I'd rather not set a cash precedent above $195K. Also — can HR confirm the equity refresh schedule we can offer?", 44),
        msg("hi003", "recruiter@acme.com", "Talent Team", "I'll check with HR on equity. But I want to be honest: $195K is likely not enough to beat Stripe. Recommend going to $200K given the 4-month search cost. Let's decide by Thursday noon so I have time to call the candidate.", 40),
    ]
))

# ── 8. MEDIUM — Quarterly review ─────────────────────────────────────────────
OUTLOOK_THREADS.append(ConversationDoc(
    platform      = "outlook",
    chat_id       = "thread_qreview_008",
    participants  = ["manager@acme.com", "you@acme.com"],
    display_names = {"manager@acme.com": "Lisa Wong (Manager)", "you@acme.com": "You"},
    channel_name  = "Q3 Performance Review — Please Complete Self-Assessment by Next Week",
    fetched_at    = ts(0),
    messages      = [
        msg("qr001", "manager@acme.com", "Lisa Wong", "Hi — just a reminder that Q3 performance reviews are due next week. Please complete your self-assessment in Workday by next Wednesday. Focus on the three key objectives we set in April, any blockers you encountered, and your development goals for Q4. Happy to discuss beforehand if helpful.", 72),
        msg("qr002", "you@acme.com", "You", "Thanks Lisa — I'll have it done by Tuesday. Should I include the cross-team collaboration work on the platform migration or keep it focused on my primary OKRs?", 68),
        msg("qr003", "manager@acme.com", "Lisa Wong", "Include both — cross-team impact is definitely relevant context. Looking forward to reading it.", 60),
    ]
))

# ── 9. MEDIUM — Cross-team dependency ────────────────────────────────────────
OUTLOOK_THREADS.append(ConversationDoc(
    platform      = "outlook",
    chat_id       = "thread_crossteam_009",
    participants  = ["platform@acme.com", "product@acme.com", "you@acme.com"],
    display_names = {"platform@acme.com": "Platform Team", "product@acme.com": "Product Team", "you@acme.com": "You"},
    channel_name  = "API Contract Change — Breaking Change Planned for v2.1 — Your Input Needed",
    fetched_at    = ts(0),
    messages      = [
        msg("ct001", "platform@acme.com", "Platform Team", "We are planning to deprecate the /v1/search endpoint in favour of /v2/search in the next release (v2.1, planned for end of quarter). The new endpoint has a breaking change in the response schema. We need to know which teams are currently using /v1/search so we can coordinate the migration. Please reply with your usage status by Thursday.", 56),
        msg("ct002", "product@acme.com", "Product Team", "We're using /v1/search in the recommendation engine and the search results page. We'll need at least 2 sprint cycles to migrate. Can you hold the deprecation until the following quarter?", 50),
        msg("ct003", "platform@acme.com", "Platform Team", "We can extend the deprecation timeline to Q1 but we'd need formal confirmation from all consumer teams by end of this week. Please reply with: (1) whether you're using the endpoint, (2) estimated migration timeline.", 44),
    ]
))

# ── 10. LOW — Company newsletter ──────────────────────────────────────────────
OUTLOOK_THREADS.append(ConversationDoc(
    platform      = "outlook",
    chat_id       = "thread_newsletter_010",
    participants  = ["hr@acme.com", "you@acme.com"],
    display_names = {"hr@acme.com": "People Team", "you@acme.com": "You"},
    channel_name  = "ACME Monthly Update — October Edition 🎉",
    fetched_at    = ts(0),
    messages      = [
        msg("nl001", "hr@acme.com", "People Team", "Hi team! Welcome to October's ACME Monthly Update. This month: new coffee machine in kitchen 3B, upcoming team building on Oct 25 (sign up on the intranet), reminder to submit your benefit elections by Oct 31, and congrats to the Sydney team for hitting their Q3 targets! Full details in the newsletter attached. Have a great weekend everyone 🎉", 96),
    ]
))

# ── 11. LOW — IT maintenance window ──────────────────────────────────────────
OUTLOOK_THREADS.append(ConversationDoc(
    platform      = "outlook",
    chat_id       = "thread_it_maint_011",
    participants  = ["it@acme.com", "you@acme.com"],
    display_names = {"it@acme.com": "IT Department", "you@acme.com": "You"},
    channel_name  = "Scheduled Maintenance: VPN Gateway Restart — Saturday 2-4 AM UTC",
    fetched_at    = ts(0),
    messages      = [
        msg("it001", "it@acme.com", "IT Department", "This is a notification of scheduled maintenance on the VPN gateway. The service will be unavailable from 2:00 AM to 4:00 AM UTC this Saturday. Impact: VPN access will be interrupted during this window. If you need VPN access during these hours, please plan accordingly. No action required unless you have concerns. Contact the IT helpdesk if you have questions.", 120),
    ]
))

# ── 12. LOW — Social event ────────────────────────────────────────────────────
OUTLOOK_THREADS.append(ConversationDoc(
    platform      = "outlook",
    chat_id       = "thread_party_012",
    participants  = ["events@acme.com", "you@acme.com"],
    display_names = {"events@acme.com": "Events Committee", "you@acme.com": "You"},
    channel_name  = "You're Invited! ACME End-of-Year Party — Dec 15 — RSVP by Nov 30",
    fetched_at    = ts(0),
    messages      = [
        msg("pa001", "events@acme.com", "Events Committee", "You're invited to the ACME End-of-Year Celebration! 🥂 Date: December 15 | Time: 7 PM | Venue: The Grand Rooftop, Downtown. Dinner, open bar, awards ceremony. Plus/ones welcome. Please RSVP on the event portal by November 30 so we can finalise catering numbers. Questions? Reply to this email or ping #events in Slack.", 240),
    ]
))

print(f"✓ Defined {len(OUTLOOK_THREADS)} Outlook test threads")
for i, t in enumerate(OUTLOOK_THREADS):
    print(f"   {i+1:2d}. [{t.chat_id}]  {len(t.messages)} msgs — {t.channel_name[:60]}...")

## Cell 4 — Teams Test Channels (7 channels)

Covers: unanswered questions, answered questions, rhetorical questions,
varying urgency, cross-channel topics, and enough volume for digest summaries.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  TEAMS CHANNELS
#  7 channels — each is one ConversationDoc with platform="teams"
#  section_heading (= channel_name) is used by all channel filters
# ════════════════════════════════════════════════════════════════════════════

TEAMS_CHANNELS = []

# ── CH1: Engineering › General ────────────────────────────────────────────────
# Contains: 3 unanswered genuine questions, 2 answered, 4 rhetorical/fake
TEAMS_CHANNELS.append(ConversationDoc(
    platform      = "teams",
    chat_id       = "ch_engineering_general",
    participants  = ["alice.j", "bob.s", "carol.t", "dave.m", "eve.r"],
    display_names = {"alice.j": "Alice Johnson", "bob.s": "Bob Smith", "carol.t": "Carol Torres", "dave.m": "Dave Miller", "eve.r": "Eve Rao"},
    channel_name  = "Engineering › General",
    fetched_at    = ts(0),
    messages      = [
        msg("eg001", "alice.j", "Alice Johnson", "Morning everyone. Quick note: the CI pipeline is taking 22 minutes on average now. Has anyone looked into why the build times doubled since last Friday?", 48, 0, "ch_engineering_general"),
        msg("eg002", "bob.s", "Bob Smith", "Yeah I noticed that too. I think it's related to the new test suite Carol added. Carol — did you add parallelisation to those integration tests?", 47, 50, "ch_engineering_general"),
        msg("eg003", "carol.t", "Carol Torres", "Yes I did. The tests are running sequentially on the CI runner because we only have 2 cores allocated. I've submitted a request to DevOps to bump it to 4 cores.", 47, 40, "ch_engineering_general"),
        msg("eg004", "alice.j", "Alice Johnson", "Carol — has DevOps responded to your request yet? We're burning 15 extra minutes per pipeline run.", 46, 0, "ch_engineering_general"),  # UNANSWERED - HIGH
        msg("eg005", "dave.m", "Dave Miller", "Separately — has anyone pushed the hotfix for the race condition in the auth module? The QA team is blocked on their test run.", 44, 0, "ch_engineering_general"),  # UNANSWERED - HIGH
        msg("eg006", "alice.j", "Alice Johnson", "Good morning all!", 43, 30, "ch_engineering_general"),  # RHETORICAL
        msg("eg007", "bob.s", "Bob Smith", "Ready to kick off the sprint review?", 42, 0, "ch_engineering_general"),  # RHETORICAL
        msg("eg008", "eve.r", "Eve Rao", "Who owns the feature flag service? I need to enable the new checkout experiment for the EU region and I can't find the config docs anywhere.", 40, 0, "ch_engineering_general"),  # UNANSWERED - MEDIUM
        msg("eg009", "carol.t", "Carol Torres", "Eve — I think Dave owns it. Dave?", 39, 30, "ch_engineering_general"),
        msg("eg010", "dave.m", "Dave Miller", "The auth hotfix PR is up — #1847. Still waiting on two approvals before I can merge.", 38, 0, "ch_engineering_general"),
        msg("eg011", "alice.j", "Alice Johnson", "Is the staging environment back up? It was down earlier.", 36, 0, "ch_engineering_general"),  # ANSWERED BELOW
        msg("eg012", "bob.s", "Bob Smith", "Yes — staging is back. Turned out to be a misconfigured health check. Fixed 10 minutes ago.", 35, 30, "ch_engineering_general"),
        msg("eg013", "eve.r", "Eve Rao", "Great call last week btw 👍", 34, 0, "ch_engineering_general"),  # RHETORICAL/SOCIAL
        msg("eg014", "carol.t", "Carol Torres", "Agree! Good progress.", 33, 0, "ch_engineering_general"),  # RHETORICAL
    ]
))

# ── CH2: Product › Roadmap ────────────────────────────────────────────────────
# Contains: 4 unanswered questions (spec deadline, prioritisation, launch date, ownership)
TEAMS_CHANNELS.append(ConversationDoc(
    platform      = "teams",
    chat_id       = "ch_product_roadmap",
    participants  = ["pm.sara", "pm.jake", "design.mia", "eng.lead.priya", "cpo.raj"],
    display_names = {"pm.sara": "Sara Kim (PM)", "pm.jake": "Jake North (PM)", "design.mia": "Mia Chen (Design)", "eng.lead.priya": "Priya Sharma (Eng Lead)", "cpo.raj": "Raj Patel (CPO)"},
    channel_name  = "Product › Roadmap",
    fetched_at    = ts(0),
    messages      = [
        msg("pr001", "pm.sara", "Sara Kim", "Team — the v2 spec for the notification centre is still in draft. Engineering says they need the finalised spec before they can start sprint planning. When is the spec going to be locked?", 72, 0, "ch_product_roadmap"),  # UNANSWERED - HIGH
        msg("pr002", "pm.jake", "Jake North", "Sara — I've been blocked on getting UX sign-off. Mia, are the notification panel designs finalised? Engineering needs the interaction model before I can write the spec.", 71, 0, "ch_product_roadmap"),  # UNANSWERED - MEDIUM
        msg("pr003", "design.mia", "Mia Chen", "The core flows are done. I still need to finalise the empty state and the badge behaviour for grouped notifications. I can have those done by Thursday.", 70, 30, "ch_product_roadmap"),
        msg("pr004", "eng.lead.priya", "Priya Sharma", "Thursday is too late for our sprint planning session on Wednesday. Can we use the current designs as a working draft and update the spec iteratively?", 69, 0, "ch_product_roadmap"),
        msg("pr005", "pm.sara", "Sara Kim", "Raj — this is the third time the notification centre has been delayed because of missing upstream dependencies. Should we deprioritise it and pull up the mobile push feature instead?", 68, 0, "ch_product_roadmap"),  # UNANSWERED - MEDIUM
        msg("pr006", "pm.jake", "Jake North", "Also — what is the launch target for Q4? The sales team told a customer we'd have this in November. Is that still the plan?", 65, 0, "ch_product_roadmap"),  # UNANSWERED - HIGH
        msg("pr007", "eng.lead.priya", "Priya Sharma", "If we start sprint planning Wednesday with the current draft spec, we can hit a November beta. But scope needs to be locked by Friday — otherwise were looking at December.", 63, 0, "ch_product_roadmap"),
        msg("pr008", "design.mia", "Mia Chen", "I can prioritise the empty state today and deliver by Wednesday morning. Will that help?", 60, 0, "ch_product_roadmap"),
        msg("pr009", "pm.jake", "Jake North", "Yes — that would unblock the spec. Let's do that.", 59, 30, "ch_product_roadmap"),
        msg("pr010", "pm.sara", "Sara Kim", "Raj — still need your call on whether to deprioritise or push through. This affects the whole team's Q4 plan.", 55, 0, "ch_product_roadmap"),  # UNANSWERED - HIGH URGENCY REPEAT
    ]
))

# ── CH3: DevOps › Incidents ───────────────────────────────────────────────────
# Contains: 5 unanswered questions (post-mortem owner, runbook, monitoring)
TEAMS_CHANNELS.append(ConversationDoc(
    platform      = "teams",
    chat_id       = "ch_devops_incidents",
    participants  = ["sre.tom", "sre.nina", "devops.kai", "eng.rachel", "vp.ops.sam"],
    display_names = {"sre.tom": "Tom Wei (SRE)", "sre.nina": "Nina Patel (SRE)", "devops.kai": "Kai Brooks (DevOps)", "eng.rachel": "Rachel Kim (Eng)", "vp.ops.sam": "Sam Torres (VP Ops)"},
    channel_name  = "DevOps › Incidents",
    fetched_at    = ts(0),
    messages      = [
        msg("di001", "sre.tom", "Tom Wei", "Last night's database failover incident — we had 14 minutes of degraded service. The auto-failover did not trigger as expected. Who is writing the post-mortem for this? It's been 18 hours and I don't see a doc started.", 22, 0, "ch_devops_incidents"),  # UNANSWERED - HIGH
        msg("di002", "sre.nina", "Nina Patel", "I thought Rachel was on it since she was the on-call engineer. Rachel — can you confirm?", 21, 30, "ch_devops_incidents"),
        msg("di003", "eng.rachel", "Rachel Kim", "I can start the post-mortem doc. But I need the CloudWatch metrics from the 2-4 AM window — Tom, do you have those? I don't have the right IAM permissions to pull them directly.", 21, 0, "ch_devops_incidents"),  # UNANSWERED - MEDIUM
        msg("di004", "devops.kai", "Kai Brooks", "Separately — why did the PagerDuty alert take 8 minutes to fire? The SLA is 2 minutes. Is there a config issue with our alerting thresholds?", 20, 0, "ch_devops_incidents"),  # UNANSWERED - HIGH
        msg("di005", "sre.tom", "Tom Wei", "Good question on PagerDuty. I think it's the evaluation window we set during the last cost-reduction exercise. We extended it from 1 min to 5 min. Kai — can you revert that change today?", 19, 30, "ch_devops_incidents"),
        msg("di006", "devops.kai", "Kai Brooks", "I can revert. But I need VP approval to change PagerDuty config — there was a policy change last month. Sam — can you approve a rollback of the evaluation window to 1 minute?", 18, 0, "ch_devops_incidents"),  # UNANSWERED - HIGH
        msg("di007", "sre.nina", "Nina Patel", "Also — does our runbook cover the RDS failover procedure? I couldn't find a clear step-by-step during the incident.", 17, 0, "ch_devops_incidents"),  # UNANSWERED - MEDIUM
        msg("di008", "eng.rachel", "Rachel Kim", "I've started the post-mortem doc: [link: acme.wiki/postmortem/2025-04-db]. Tom — still need those CloudWatch metrics.", 15, 0, "ch_devops_incidents"),
        msg("di009", "sre.tom", "Tom Wei", "I'll get you the metrics within the hour. Also I've confirmed the auto-failover didn't trigger because the health check endpoint was timing out but not returning a definitive error — it was returning 408 which our monitoring treated as transient.", 14, 0, "ch_devops_incidents"),
        msg("di010", "vp.ops.sam", "Sam Torres", "Thanks for the update. This is unacceptable — we need the runbook updated and the monitoring fixed before end of week. Who is owning each action item?", 12, 0, "ch_devops_incidents"),  # UNANSWERED - HIGH (ownership unclear)
    ]
))

# ── CH4: Design › General ──────────────────────────────────────────────────────
# Contains: 2 unanswered questions, 3 answered, lots of design discussion
TEAMS_CHANNELS.append(ConversationDoc(
    platform      = "teams",
    chat_id       = "ch_design_general",
    participants  = ["design.mia", "design.felix", "design.aisha", "pm.sara", "brand.leo"],
    display_names = {"design.mia": "Mia Chen", "design.felix": "Felix Park", "design.aisha": "Aisha Williams", "pm.sara": "Sara Kim (PM)", "brand.leo": "Leo Martínez (Brand)"},
    channel_name  = "Design › General",
    fetched_at    = ts(0),
    messages      = [
        msg("dg001", "design.mia", "Mia Chen", "Sharing the new design system tokens for review. I've updated the primary button from #1565C0 to #185FA5 for better WCAG AA contrast. Does everyone agree with this change? It affects 240 components.", 36, 0, "ch_design_general"),
        msg("dg002", "design.felix", "Felix Park", "Looks good to me. The contrast ratio goes from 3.8 to 4.7 — passes AA. Does it still match our brand guidelines Leo?", 35, 30, "ch_design_general"),  # ANSWERED BELOW
        msg("dg003", "brand.leo", "Leo Martínez", "Yes — the darker blue is actually closer to our original brand specification. The lighter version was a mistake from 2022. I'm approving this change.", 35, 0, "ch_design_general"),
        msg("dg004", "design.aisha", "Aisha Williams", "How long will it take to propagate this change across the component library? Will it break anything in the existing Storybook?", 34, 0, "ch_design_general"),  # ANSWERED BELOW
        msg("dg005", "design.mia", "Mia Chen", "Aisha — I've already tested it in Storybook. The only visual change is the button colour itself. It'll take about 2 hours to propagate once Felix does the Figma token update.", 33, 30, "ch_design_general"),
        msg("dg006", "design.felix", "Felix Park", "I'll update the Figma tokens this afternoon. Should be live by 5 PM.", 33, 0, "ch_design_general"),
        msg("dg007", "pm.sara", "Sara Kim", "Who is responsible for coordinating with Engineering to get the updated tokens into the codebase before the next release?", 30, 0, "ch_design_general"),  # UNANSWERED - MEDIUM
        msg("dg008", "design.aisha", "Aisha Williams", "Also — are we updating the dark mode variants at the same time, or is that a separate ticket?", 28, 0, "ch_design_general"),  # UNANSWERED - LOW
        msg("dg009", "design.mia", "Mia Chen", "Good call everyone — nice teamwork 🎉", 26, 0, "ch_design_general"),  # RHETORICAL/SOCIAL
        msg("dg010", "brand.leo", "Leo Martínez", "Agreed! Really smooth process today.", 25, 30, "ch_design_general"),  # RHETORICAL
        msg("dg011", "design.felix", "Felix Park", "Anyone want to grab coffee before the 3pm? ☕", 24, 0, "ch_design_general"),  # SOCIAL/RHETORICAL
    ]
))

# ── CH5: Leadership › Strategy ────────────────────────────────────────────────
# Contains: 3 unanswered strategic questions (ownership, timeline, decision)
TEAMS_CHANNELS.append(ConversationDoc(
    platform      = "teams",
    chat_id       = "ch_leadership_strategy",
    participants  = ["ceo.sarah", "cto.alex", "cfo.michael", "cpo.raj", "vp.sales.jenny"],
    display_names = {"ceo.sarah": "Sarah Chen (CEO)", "cto.alex": "Alex Ramos (CTO)", "cfo.michael": "Michael Torres (CFO)", "cpo.raj": "Raj Patel (CPO)", "vp.sales.jenny": "Jenny Liu (VP Sales)"},
    channel_name  = "Leadership › Strategy",
    fetched_at    = ts(0),
    messages      = [
        msg("ls001", "ceo.sarah", "Sarah Chen", "Following up from today's board meeting. The board has asked us to provide a formal response to the acquisition interest from TechCo by next Friday. Raj — what is our product differentiation story for the next 18 months? They'll want to see this.", 24, 0, "ch_leadership_strategy"),  # UNANSWERED - HIGH
        msg("ls002", "cfo.michael", "Michael Torres", "Sarah — do we have a view on valuation? The board mentioned $340M but I want to understand if that's based on current ARR multiples or growth-adjusted. This affects whether we even engage.", 23, 30, "ch_leadership_strategy"),  # UNANSWERED - HIGH
        msg("ls003", "cto.alex", "Alex Ramos", "From a technical DD perspective — we need to clean up three legacy services before any serious due diligence. If TechCo does a code review, those will raise red flags. Do we have runway to do this in parallel while evaluating the offer?", 22, 0, "ch_leadership_strategy"),  # UNANSWERED - MEDIUM
        msg("ls004", "vp.sales.jenny", "Jenny Liu", "I'd like to flag that we have two large enterprise prospects — $2M ARR combined — currently in final negotiation. An acquisition rumour could spook them. How are we managing information leakage risk?", 20, 0, "ch_leadership_strategy"),  # UNANSWERED - HIGH
        msg("ls005", "ceo.sarah", "Sarah Chen", "Good points all around. Let's do a dedicated session Thursday to align. Michael — can you model three valuation scenarios before then?", 18, 0, "ch_leadership_strategy"),
        msg("ls006", "cfo.michael", "Michael Torres", "I'll have three scenarios — bear/base/bull — ready by Wednesday EOD so everyone has time to review before Thursday.", 17, 30, "ch_leadership_strategy"),
        msg("ls007", "cpo.raj", "Raj Patel", "I'll prepare the 18-month roadmap summary. Sarah — do you want this framed as-is or acquisition-context adjusted?", 16, 0, "ch_leadership_strategy"),
        msg("ls008", "ceo.sarah", "Sarah Chen", "Both — two versions. Authentic roadmap + what it looks like post-integration. Raj — can you have both ready for Thursday?", 15, 0, "ch_leadership_strategy"),
        msg("ls009", "cpo.raj", "Raj Patel", "Will do. Thursday morning.", 14, 30, "ch_leadership_strategy"),
    ]
))

# ── CH6: Customer Success › All ───────────────────────────────────────────────
# Contains: 3 unanswered customer-facing questions
TEAMS_CHANNELS.append(ConversationDoc(
    platform      = "teams",
    chat_id       = "ch_customer_success",
    participants  = ["cs.lead.tom", "cs.amy", "cs.ben", "eng.support.rachel"],
    display_names = {"cs.lead.tom": "Tom Bradley (CS Lead)", "cs.amy": "Amy Park (CS)", "cs.ben": "Ben Okonkwo (CS)", "eng.support.rachel": "Rachel Kim (Eng Support)"},
    channel_name  = "Customer Success › All",
    fetched_at    = ts(0),
    messages      = [
        msg("cu001", "cs.amy", "Amy Park", "BigCorp just opened a P1 ticket — their SSO integration is broken after the latest update. They have a company-wide login issue affecting 800 users. Do we have an ETA on a fix?", 8, 0, "ch_customer_success"),  # UNANSWERED - HIGH
        msg("cu002", "cs.lead.tom", "Tom Bradley", "Rachel — Engineering — can you look at this immediately? BigCorp is $180K ARR and they're already unhappy from last month's outage.", 7, 45, "ch_customer_success"),
        msg("cu003", "eng.support.rachel", "Rachel Kim", "I'm looking now. The SSO issue appears to be related to the SAML assertion timestamp validation change we pushed in v2.3.1. Did BigCorp update to 2.3.1 today?", 7, 30, "ch_customer_success"),
        msg("cu004", "cs.amy", "Amy Park", "Yes — they auto-updated. Can we roll them back to 2.3.0?", 7, 0, "ch_customer_success"),  # UNANSWERED - HIGH
        msg("cu005", "cs.ben", "Ben Okonkwo", "Separately — Pinnacle Tech asked if we support SCIM provisioning. Do we? I couldn't find a clear answer in the docs.", 6, 0, "ch_customer_success"),  # UNANSWERED - MEDIUM
        msg("cu006", "cs.lead.tom", "Tom Bradley", "Great question Ben — Rachel, do we support SCIM?", 5, 30, "ch_customer_success"),
        msg("cu007", "eng.support.rachel", "Rachel Kim", "I've confirmed the SAML issue. I'm writing a hotfix now — ETA 45 minutes for a patch. Amy — can you message BigCorp to let them know we have a fix in progress? I'll post a workaround in 10 minutes.", 5, 0, "ch_customer_success"),
        msg("cu008", "cs.amy", "Amy Park", "On it. Also — do we have a status page update we can point them to?", 4, 30, "ch_customer_success"),  # UNANSWERED - MEDIUM
        msg("cu009", "cs.ben", "Ben Okonkwo", "Tom — are you flagging this to the account team for BigCorp? Given the history, they might want exec outreach.", 3, 0, "ch_customer_success"),
        msg("cu010", "cs.lead.tom", "Tom Bradley", "Yes — I'm drafting an email to their CTO now. Rachel — once the hotfix is ready, please confirm the version number so I can include it.", 2, 30, "ch_customer_success"),
    ]
))

# ── CH7: General › Company ────────────────────────────────────────────────────
# High noise channel — lots of rhetorical/social messages, few real questions
TEAMS_CHANNELS.append(ConversationDoc(
    platform      = "teams",
    chat_id       = "ch_general_company",
    participants  = ["hr.team", "alice.j", "bob.s", "carol.t", "dave.m", "eve.r", "design.mia", "pm.sara"],
    display_names = {"hr.team": "People Team", "alice.j": "Alice Johnson", "bob.s": "Bob Smith", "carol.t": "Carol Torres", "dave.m": "Dave Miller", "eve.r": "Eve Rao", "design.mia": "Mia Chen", "pm.sara": "Sara Kim"},
    channel_name  = "General › Company",
    fetched_at    = ts(0),
    messages      = [
        msg("gc001", "hr.team", "People Team", "Good morning team! 🌟 Don't forget all-hands is this Thursday at 10 AM in the main auditorium and Zoom for remote folks. Submit your questions for the Q&A by Wednesday noon.", 96, 0, "ch_general_company"),
        msg("gc002", "alice.j", "Alice Johnson", "Morning! Can't wait for all-hands 😊", 95, 30, "ch_general_company"),  # RHETORICAL/SOCIAL
        msg("gc003", "bob.s", "Bob Smith", "What time does the free lunch start? 🍕", 95, 0, "ch_general_company"),  # GENUINE but low urgency
        msg("gc004", "hr.team", "People Team", "Bob — lunch is at 12:30 PM after the all-hands Q&A session!", 94, 30, "ch_general_company"),
        msg("gc005", "carol.t", "Carol Torres", "Is there a vegetarian option? I forgot to flag that in the survey.", 94, 0, "ch_general_company"),  # GENUINE - LOW
        msg("gc006", "hr.team", "People Team", "Yes — there's always a full vegetarian spread Carol. No need to worry!", 93, 30, "ch_general_company"),
        msg("gc007", "dave.m", "Dave Miller", "Excited!! Can't believe it's been a year already 🚀", 92, 0, "ch_general_company"),  # RHETORICAL/SOCIAL
        msg("gc008", "eve.r", "Eve Rao", "Same!! Time flies.", 91, 30, "ch_general_company"),  # RHETORICAL
        msg("gc009", "design.mia", "Mia Chen", "Will the all-hands be recorded for those of us in different time zones who can't make it live?", 90, 0, "ch_general_company"),  # GENUINE - LOW URGENCY, UNANSWERED
        msg("gc010", "pm.sara", "Sara Kim", "Great question! I also want to know this.", 89, 30, "ch_general_company"),  # RHETORICAL
        msg("gc011", "alice.j", "Alice Johnson", "Ready? 🙌", 88, 0, "ch_general_company"),  # RHETORICAL
        msg("gc012", "hr.team", "People Team", "Has anyone seen the new office plants in the 4th floor kitchen? They look amazing! 🌿", 86, 0, "ch_general_company"),  # RHETORICAL
        msg("gc013", "bob.s", "Bob Smith", "Haha yes! Very fancy 😄", 85, 30, "ch_general_company"),  # SOCIAL
        msg("gc014", "carol.t", "Carol Torres", "Does HR know if the gym subsidy is being extended into Q1 next year? I want to plan my membership renewal.", 84, 0, "ch_general_company"),  # GENUINE - LOW, UNANSWERED
        msg("gc015", "hr.team", "People Team", "Thanks for a great week everyone! Looking forward to all-hands Thursday 🎉", 80, 0, "ch_general_company"),  # RHETORICAL
    ]
))

print(f"✓ Defined {len(TEAMS_CHANNELS)} Teams channels")
for i, ch in enumerate(TEAMS_CHANNELS):
    print(f"   {i+1}. [{ch.chat_id}]  {len(ch.messages)} messages — {ch.channel_name}")

## Cell 5 — Seeder functions

`seed_outlook_threads()` and `seed_teams_channels()` each call the existing
`ChatDocumentIndexer.index_conversation()` — **no main app code is modified**.
A dry-run mode lets you inspect what will be written before committing.

In [ ]:
import time

async def seed_outlook_threads(threads=OUTLOOK_THREADS, dry_run=False):
    """
    Push all Outlook test threads into SeaweedFS + LanceDB.
    Calls ChatDocumentIndexer.index_conversation() — identical to the live connector path.
    """
    print(f"{'[DRY RUN] ' if dry_run else ''}Seeding {len(threads)} Outlook threads...")
    results = []
    for i, thread in enumerate(threads):
        if dry_run:
            fid = hashlib.sha256(thread.document_key().encode()).hexdigest()[:16]
            print(f"  [{i+1:2d}] SKIP (dry-run) — {thread.channel_name[:55]}...  file_id={fid}")
            results.append({"file_id": fid, "thread": thread.chat_id, "status": "dry-run"})
            continue

        t0 = time.time()
        try:
            file_id = await indexer.index_conversation(thread)
            elapsed = time.time() - t0
            print(f"  [{i+1:2d}] ✓ {thread.channel_name[:55]}...  file_id={file_id}  ({elapsed:.1f}s)")
            results.append({"file_id": file_id, "thread": thread.chat_id, "status": "ok", "elapsed": elapsed})
        except Exception as e:
            elapsed = time.time() - t0
            print(f"  [{i+1:2d}] ✗ FAILED: {thread.chat_id} — {e}")
            results.append({"file_id": None, "thread": thread.chat_id, "status": "error", "error": str(e)})

    ok  = sum(1 for r in results if r["status"] == "ok")
    err = sum(1 for r in results if r["status"] == "error")
    print(f"\nOutlook seed complete: {ok} ok, {err} errors, {len(threads)} total")
    return results


async def seed_teams_channels(channels=TEAMS_CHANNELS, dry_run=False):
    """
    Push all Teams test channels into SeaweedFS + LanceDB.
    """
    print(f"{'[DRY RUN] ' if dry_run else ''}Seeding {len(channels)} Teams channels...")
    results = []
    for i, channel in enumerate(channels):
        if dry_run:
            fid = hashlib.sha256(channel.document_key().encode()).hexdigest()[:16]
            print(f"  [{i+1}] SKIP (dry-run) — {channel.channel_name}  file_id={fid}")
            results.append({"file_id": fid, "channel": channel.chat_id, "status": "dry-run"})
            continue

        t0 = time.time()
        try:
            file_id = await indexer.index_conversation(channel)
            elapsed = time.time() - t0
            print(f"  [{i+1}] ✓ {channel.channel_name}  file_id={file_id}  ({elapsed:.1f}s)")
            results.append({"file_id": file_id, "channel": channel.chat_id, "status": "ok", "elapsed": elapsed})
        except Exception as e:
            elapsed = time.time() - t0
            print(f"  [{i+1}] ✗ FAILED: {channel.chat_id} — {e}")
            results.append({"file_id": None, "channel": channel.chat_id, "status": "error", "error": str(e)})

    ok  = sum(1 for r in results if r["status"] == "ok")
    err = sum(1 for r in results if r["status"] == "error")
    print(f"\nTeams seed complete: {ok} ok, {err} errors, {len(channels)} total")
    return results


print("✓ Seeder functions ready. Run Cell 6 to seed data.")

## Cell 6 — Dry run (inspect what will be written)

In [ ]:
# ── Dry run — shows what WILL be written without touching SeaweedFS or LanceDB ──
print("=" * 65)
print("DRY RUN — Outlook")
print("=" * 65)
await seed_outlook_threads(dry_run=True)

print()
print("=" * 65)
print("DRY RUN — Teams")
print("=" * 65)
await seed_teams_channels(dry_run=True)

## Cell 7 — Real seed (writes to SeaweedFS + LanceDB)

⚠️ **This cell makes real writes.** Run Cell 6 first to verify the data.
Embeddings are generated via the OpenRouter API — ~60 API calls total.

In [ ]:
# ── REAL SEED ─────────────────────────────────────────────────────────────────
# Comment out either block if you only want to seed one platform.
embedder = OpenRouterEmbedder(api_key="")

print("Seeding Outlook threads...")
outlook_results = await seed_outlook_threads(dry_run=False)

print()
print("Seeding Teams channels...")
teams_results = await seed_teams_channels(dry_run=False)

print()
print("=" * 65)
print("SEED COMPLETE")
print("=" * 65)

## Cell 8 — Verify what landed in LanceDB

In [ ]:
import pandas as pd

# Reload table references after writes
store.refresh()

try:
    chunk_df = store.chunks.to_pandas()
    doc_df   = store.documents.to_pandas()
except Exception as e:
    print("Could not load tables:", e)
    chunk_df = pd.DataFrame()
    doc_df   = pd.DataFrame()

print("=" * 65)
print(f"  TOTAL DOCUMENTS : {len(doc_df)}")
print(f"  TOTAL CHUNKS    : {len(chunk_df)}")
print("=" * 65)

if len(chunk_df):
    print("\n── Chunks by platform ──")
    print(chunk_df.groupby("platform").size().rename("chunk_count").to_string())

    print("\n── Outlook threads ──")
    ol = chunk_df[chunk_df["platform"] == "outlook"]
    if len(ol):
        for tid, grp in ol.groupby("thread_id"):
            subj = grp["section_heading"].iloc[0] if "section_heading" in grp.columns else grp["filename"].iloc[0]
            print(f"  {tid[:20]}...  {len(grp):2d} chunks  |  {str(subj)[:55]}")

    print("\n── Teams channels ──")
    tm = chunk_df[chunk_df["platform"] == "teams"]
    if len(tm):
        for ch, grp in tm.groupby("section_heading"):
            print(f"  {str(ch):<40}  {len(grp):2d} chunks")

    print("\n── sent_at range ──")
    print("  oldest:", chunk_df["sent_at"].min())
    print("  newest:", chunk_df["sent_at"].max())

print("\n✓ Verification complete — your intelligence endpoints are ready to test.")

## Cell 9 — Quick intelligence sanity checks

Runs your actual intelligence functions directly — the same code the API routes call.
No HTTP server needed.

## Cell 10 — Targeted edge-case tests

Tests specific intelligence behaviours: channel filter, urgency classification,
rhetorical filtering, and time-window lookback.

## Cell 11 — Re-seed a single item (useful for iteration)

Use this to update one thread or channel without re-running the full seed.

## Cell 12 — Cleanup (optional — deletes all seeded test data)

In [ ]:
# !! WARNING — This deletes ALL seeded test records. Uncomment to run. !!

# SEEDED_FILE_IDS = [
#     hashlib.sha256(t.document_key().encode()).hexdigest()[:16]
#     for t in OUTLOOK_THREADS + TEAMS_CHANNELS
# ]
#
# for fid in SEEDED_FILE_IDS:
#     try:
#         store.chunks.delete(f'file_id = "{fid}"')
#         store.documents.delete(f'file_id = "{fid}"')
#         print(f"  Deleted {fid}")
#     except Exception as e:
#         print(f"  Failed {fid}: {e}")
#
# store.rebuild_fts_index()
# print("\n✓ Cleanup complete")

print("Cleanup cell is commented out for safety. Uncomment to run.")

In [ ]:
# import sys, asyncio
# from datetime import datetime, timedelta, timezone
# from pathlib  import Path

# APP_ROOT = Path("app")
# sys.path.insert(0, str(APP_ROOT.resolve()))

# import lancedb
# from connectors.chat_indexer import ChatDocumentIndexer
# from connectors.models       import ConversationDoc, MessageTurn
# from ingestion.indexer       import OpenRouterEmbedder
# from storage.lancedb_store   import LanceDBStore
# from storage.seaweed         import SeaweedStore
# from config                  import OPENROUTER_KEY

# _db_conn  = lancedb.connect("./data/lancedb")
# _store    = LanceDBStore(_db_conn)
# _seaweed  = SeaweedStore("http://localhost:8888")
# _embedder = OpenRouterEmbedder(api_key=OPENROUTER_KEY)
# _indexer  = ChatDocumentIndexer(store=_store, embedder=_embedder, seaweed=_seaweed)

# def _ts(h, m=0):
#     return (datetime.now(timezone.utc) - timedelta(hours=h, minutes=m)).strftime("%Y-%m-%dT%H:%M:%S")

# def _m(mid, sid, name, content, h, m=0):
#     return MessageTurn(message_id=mid, sender_id=sid, sender_name=name,
#                        content=content, sent_at=_ts(h, m), thread_id=None)

# EXTRA = [

#   # ── E1 · CRITICAL — PII data leak, GDPR 72h clock ───────────────────────────
#   ConversationDoc(
#     platform="outlook", chat_id="thread_extra_data_leak_101",
#     participants=["security@acme.com","cto@acme.com","you@acme.com","legal@acme.com"],
#     display_names={"security@acme.com":"Security Team","cto@acme.com":"Raj Patel (CTO)","you@acme.com":"You","legal@acme.com":"Legal Counsel"},
#     channel_name="URGENT — PII Data Exposure in Prod S3 Bucket — Immediate Action Required",
#     fetched_at=_ts(0),
#     messages=[
#       _m("dl001","security@acme.com","Security Team",
#          "CRITICAL: Our scanner flagged a publicly accessible S3 bucket 'acme-prod-exports-2024' "
#          "containing raw customer PII (names, emails, partial card numbers). Bucket was public ~11 hours. "
#          "Access is blocked but full audit needed. GDPR Article 33 gives us 72 hours to notify the regulator. "
#          "Raj — need your sign-off to engage IR vendor NOW. Legal — confirm notification obligations. This cannot wait.",2),
#       _m("dl002","legal@acme.com","Legal Counsel",
#          "Confirmed — GDPR clock started when we became aware. Deadline Thursday 08:00 UTC. "
#          "Need: (1) exact record count exposed, (2) data categories, (3) remediation steps. "
#          "Security — can you have these by midnight tonight?",1,30),
#       _m("dl003","cto@acme.com","Raj Patel",
#          "IR vendor approved — engage immediately. All hands on this. "
#          "Call at 14:00 UTC today: Security, Legal, Engineering leads. "
#          "No external comms without Legal sign-off. Confirm attendance ASAP.",1,10),
#     ]
#   ),

#   # ── E2 · CRITICAL — enterprise SLA breach, $1.2M ARR churn risk ─────────────
#   ConversationDoc(
#     platform="outlook", chat_id="thread_extra_sla_breach_102",
#     participants=["enterprise.cso@globalbank.com","cs-lead@acme.com","you@acme.com","vp-sales@acme.com"],
#     display_names={"enterprise.cso@globalbank.com":"Nina Hartmann (GlobalBank CSO)","cs-lead@acme.com":"CS Lead","you@acme.com":"You","vp-sales@acme.com":"Marcus Bell (VP Sales)"},
#     channel_name="CONTRACT SLA BREACH — GlobalBank ($1.2M ARR) — Response Required by 17:00 Today",
#     fetched_at=_ts(0),
#     messages=[
#       _m("sl001","enterprise.cso@globalbank.com","Nina Hartmann",
#          "Your platform missed the 99.9% uptime SLA for the second consecutive month (99.2% in October). "
#          "Per Section 8.3 this triggers a 15% service credit AND termination rights with 30 days notice. "
#          "I'm presenting options to our board Friday. Need a written remediation plan from your exec team "
#          "by 17:00 today or we exercise the termination clause.",5),
#       _m("sl002","cs-lead@acme.com","CS Lead",
#          "Nina — sincerest apologies. Escalated to VP Sales and engineering leadership. "
#          "Marcus — this needs VP-level response today. Account is $1.2M ARR.",4,10),
#       _m("sl003","vp-sales@acme.com","Marcus Bell",
#          "On it — calling Nina personally by 15:00. Engineering — I need the RCA and mitigation "
#          "timeline in my inbox by 14:00 TODAY. This account cannot churn. Please confirm.",3,45),
#     ]
#   ),

#   # ── E3 · HIGH — competing offer expiring tomorrow ────────────────────────────
#   ConversationDoc(
#     platform="outlook", chat_id="thread_extra_offer_expiry_103",
#     participants=["talent@acme.com","eng-director@acme.com","you@acme.com"],
#     display_names={"talent@acme.com":"Talent Acquisition","eng-director@acme.com":"Sophie Okafor (Eng Director)","you@acme.com":"You"},
#     channel_name="RE: Principal Engineer Offer — Competing Offer from Anthropic — Expires Tomorrow 18:00",
#     fetched_at=_ts(0),
#     messages=[
#       _m("oe001","talent@acme.com","Talent Acquisition",
#          "Candidate for Principal Engineer (6-month search, $220K band) received offer from Anthropic "
#          "at $240K + significant RSU grant. They've given us until tomorrow 18:00 to match or improve. "
#          "Sophie — do we have budget flexibility to go to $235K? Only candidate who passed the technical bar.",8),
#       _m("oe002","eng-director@acme.com","Sophie Okafor",
#          "This hire is critical for Q1 platform roadmap. Willing to go to $232K if Finance approves tonight. "
#          "Can someone loop in HRBP and get CFO sign-off on the comp exception?",6),
#       _m("oe003","talent@acme.com","Talent Acquisition",
#          "Reaching out to CFO office now. Need final number by 09:00 tomorrow so I can call the candidate "
#          "before they decide. Sophie — please confirm tonight.",5),
#     ]
#   ),

#   # ── E4 · MEDIUM — sprint planning, soft deadline ─────────────────────────────
#   ConversationDoc(
#     platform="outlook", chat_id="thread_extra_sprint_plan_104",
#     participants=["pm@acme.com","you@acme.com","design@acme.com"],
#     display_names={"pm@acme.com":"Aiko Tanaka (PM)","you@acme.com":"You","design@acme.com":"Design Team"},
#     channel_name="Sprint 42 Planning Doc — Please Add Estimates by End of Week",
#     fetched_at=_ts(0),
#     messages=[
#       _m("sp001","pm@acme.com","Aiko Tanaka",
#          "Sprint 42 planning doc is ready in Notion. Could you add t-shirt estimates (S/M/L/XL) "
#          "to each story by Friday? Nothing needs to be precise — just enough to help scope decisions.",48),
#       _m("sp002","design@acme.com","Design Team","Final mocks in Figma by Thursday.",44),
#       _m("sp003","pm@acme.com","Aiko Tanaka","No rush if Friday is tight — planning ceremony is Monday morning.",40),
#     ]
#   ),

#   # ── E5 · LOW — office plant rota, no action needed ───────────────────────────
#   ConversationDoc(
#     platform="outlook", chat_id="thread_extra_plants_105",
#     participants=["office-mgr@acme.com","you@acme.com"],
#     display_names={"office-mgr@acme.com":"Office Manager","you@acme.com":"You"},
#     channel_name="November Plant-Watering Rota — You're Down for Week 3 (Nov 18–22)",
#     fetched_at=_ts(0),
#     messages=[
#       _m("pl001","office-mgr@acme.com","Office Manager",
#          "You're on the plant-watering rota for Week 3 (Nov 18–22). Watering can is under the kitchen sink. "
#          "The ficus by the window needs water twice that week. Reply if you'll be OOO and I'll swap you. "
#          "No action needed now 🌿",168),
#     ]
#   ),
# ]

# for doc in EXTRA:
#     fid = await _indexer.index_conversation(doc)
#     print(f"✓ {doc.chat_id}  →  fid={fid}  |  {doc.channel_name[:60]}")

# #await _seed()
# print("\nDone. Test: GET /intelligence/outlook/priority?hours=24&top_n=5")
# print("Expected order: E1 (data leak) → E2 (SLA breach) → E3 (offer) → E4 → E5")